In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
!export OMP_NUM_THREADS=4

In [2]:
model_path ='../model/Qwen2.5-Math-1.5B'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    attn_implementation='flash_attention_2',
    trust_remote_code=True,
).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

You are attempting to use Flash Attention 2.0 with a model not initialized on GPU. Make sure to move the model to GPU after initializing it on CPU with `model.to('cuda')`.


## 前置函数 
+ run_tokenize_prompt_and_output获取prompt,mask和labels
+ run_get_response_log_probs：给定inputs和labels,输出outputs的log_prob
+ run_sft_microbatch_train_step：给定log_prob和mask,进行mask求和与梯度累积
## 训练配置文件 
## 数据集 ✔
+ SFTDataset
+ 迭代器:返回一个batch,循环迭代
## 优化器
+ Adam 
+ 梯度裁剪
+ cosine学习率 ✔
## SFTTrainer
### 训练函数
+ train
+ train_step
### 评估函数
+ 评估在测试集上的表现
+ 加载vllm示例在另一个GPU上进行推理
+ 使用wanbd进行记录

## 训练函数


In [2]:
from torch.utils.data import Dataset, DataLoader
from cs336_alignment.sft import tokenize_prompt_and_output
from utils import *
class SFTDataset(Dataset):
    def __init__(
        self,
        json_path:str,
        prompt_template_path:str,
    ):  
        self.json_path = json_path
        self.template = load_template(prompt_template_path)
        self.prompts = get_r1_prompts(self.json_path,self.template)
        self.ground_truths = get_r1_ground_truths_with_template(self.json_path)
        assert len(self.prompts) == len(self.ground_truths)
    def __len__(self):
        return len(self.prompts)
    
    def __getitem__(self, idx):
        return self.prompts[idx], self.ground_truths[idx]
def sft_collate_fn(batch,tokenizer):
    prompts, ground_truths = zip(*batch)
    return tokenize_prompt_and_output(
        prompt_strs = list(prompts),
        output_strs = list(ground_truths),
        tokenizer=tokenizer
    )


INFO 03-02 13:57:42 __init__.py:190] Automatically detected platform cuda.


In [3]:
dataset = SFTDataset(
    json_path = '../preprocessed/gsm8k/test.jsonl',
    prompt_template_path='prompts/r1_zero.prompt',
)
dataset.prompts[0],dataset.ground_truths[0]

("A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?\nAssistant: <think>",
 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.</think><answer>18</answer>')

In [4]:
dataloader = DataLoader(
    dataset,
    batch_size = 4,
    shuffle = True,
    collate_fn = lambda batch: sft_collate_fn(batch,tokenizer),
    drop_last = False,
)
for x in dataloader:
    input_ids = x['input_ids']
    labels = x['labels']
    response_mask = x['response_mask']
    print(input_ids.shape)
    print(labels.shape)
    print(response_mask.shape)
    break

NameError: name 'tokenizer' is not defined

cosine学习率

In [ ]:
import math
# cosine 动态学习率
def get_lr_cosine_schedule_with_warmup(
    it,
    max_lr,
    min_lr,
    warmup_iters,
    cosine_schedule_iters,
):
    # 1. warmup 阶段
    if it < warmup_iters:
        return max_lr * it / warmup_iters

    # 2. 退火结束后
    if it > cosine_schedule_iters:
        return min_lr

    # 3. cosine decay 阶段
    decay_ratio = (it - warmup_iters) / (cosine_schedule_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    lr = min_lr + coeff * (max_lr - min_lr)
    return lr

In [ ]:
def log_generation(
        
):

In [ ]:
from dataclasses import dataclass
from cs336_alignment.sft import *
import random
@dataclass
class SFTConfig:
    pass
class SFTTrainer:
    def __init__(
        self,
        model: AutoModelForCausalLM,
        tokenizer : AutoTokenizer,
        optimizer : torch.optim.Optimizer,
        config: SFTConfig
    ):
        self.llm = model
        self.tokenizer = tokenizer
        self.optimizer = optimizer
        self.config =  config

        self.train_dataset =  SFTDataset(
            json_path = config.train_dataset_path,
            prompt_template_path=config.prompt_template_path,
        )

        self.test_dataset = SFTDataset(
            json_path = config.test_dataset_path,
            prompt_template_path=config.prompt_template_path,
        )

        # 将train_dataset保存为内存中,使用sft_collate_fn加载到GPU中
        self.dataloader = DataLoader(
            dataset = self.train_dataset,
            batch_size = config.batch_size ,
            shuffle = True,
            drop_last = False,
            collate_fn= lambda batch: sft_collate_fn(batch,self.tokenizer)
        )
    @torch.no_grad()
    def sample_responses(self):
        sampled_prompts = random.sample(self.test_dataset.prompts, self.config.sample_size)
        sampled_ground_truths = random.sample(self.test_dataset.ground_truths, self.config.sample_size)
        inputs = tokenize_prompt_and_output(
            prompt_strs = sampled_prompts,
            output_strs = sampled_ground_truths,
            tokenizer=self.tokenizer
        )
        input_ids = inputs['input_ids'].to(device)
        mask = inputs['response_mask'].to(device)
        labels = inputs['labels'].to(device)
        
    def update_lr(self,it:int,optimizer:torch.optim.Optimizer):
        lr = get_lr_cosine_schedule_with_warmup(
            iter,
            self.config.max_lr,
            self.config.min_lr,
            self.config.warmup_iters,
            self.config.cosine_schedule_iters
        )
        for param_group in optimizer.param_groups:
            param_group['lr']=lr

# + run_tokenize_prompt_and_output获取prompt,mask和labels
# + run_get_response_log_probs：给定inputs和labels,输出outputs的log_prob
# + run_sft_microbatch_train_step：给定log_prob和mask,进行mask求和与梯度累积
    def train_step(self)->dict[str:float,str:float]:
    # 每一个梯度累积算一次step
        # eval metrics
        # total_batch_loss = 0.0
        # total_batch_entropy = 0.0

        for step in range(self.config.gradient_accumulation_steps):
            data = next(self.dataloader)
            input_ids = data['input_ids'].to(device)
            labels = data['labels'].to(device)
            response_mask = data['response_mask'].to(device)

            log_probs_entropy = get_response_log_probs(
                model = self.llm,
                input_ids = input_ids,
                labels = labels,
                return_token_entropy= True,
            )

            policy_log_probs = log_probs_entropy['log_probs']
            entropy = log_probs_entropy['token_entropy'] # 注意这里没有mask

            scaled_loss,metadata = sft_microbatch_train_step(
                policy_log_probs= policy_log_probs,
                response_mask = response_mask,
                gradient_accumulation_steps= self.config.gradient_accumulation_steps,
                normalize_constant=self.config.normalize_constant,
            )# 已经反向传播

            # # eval metrics
            # total_batch_loss += scaled_loss.item()
            # total_batch_entropy += entropy.mean().item()

        avg_batch_loss = total_batch_loss / (self.config.gradient_accumulation_steps+1e-6)
        avg_batch_entropy = total_batch_entropy / (self.config.gradient_accumulation_steps+1e-6)


        torch.nn.utils.clip_grad_norm_(
                self.llm.parameters(),
                max_norm = self.config.max_grad_norm,
                norm_type=2
            )
        self.optimizer.step()
        self.optimizer.zero_grad(set_to_none=True)

        return {
            'avg_batch_entropy':avg_batch_entropy,
            'avg_batch_loss':avg_batch_loss,
        }

    
    def train(self):
        log_dict = {}
        for iter in range(self.config.start_iters,self.config.max_iters):
            self.llm.train()
            self.update_lr(iter,self.optimizer)
            train_step_log = self.train_step()

            log_dict ['train_avg_loss'] = train_step_log['avg_batch_loss']
            log_dict ['train_avg_entropy'] = train_step_log['avg_batch_entropy']

            if iter % self.config.eval_interval == 0 or iter == self.config.max_iters-1:
                



 



                


        
